# 00. Imports and Global Variables

In [56]:
import pandas as pd
import vertexai
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import List

PROJECT_ID = "financial-rag-benchmark"
LOCATION = "us-central1"
BATCH_SIZE = 10

# 01. Raw Data

In [2]:
splits = {'train': 'data/FinQA/train/metadata.jsonl', 'dev': 'data/FinQA/dev/metadata.jsonl', 'test': 'data/FinQA/test/metadata.jsonl'}
finqa = pd.read_json("hf://datasets/G4KMU/t2-ragbench/" + splits["train"], lines=True)

convfinqa = pd.read_json("hf://datasets/G4KMU/t2-ragbench/data/ConvFinQA/turn_0.jsonl", lines=True)

splits = {'train': 'data/TAT-DQA/train/metadata.jsonl', 'dev': 'data/TAT-DQA/dev/metadata.jsonl', 'test': 'data/TAT-DQA/test/metadata.jsonl'}
train = pd.read_json("hf://datasets/G4KMU/t2-ragbench/" + splits["train"], lines=True)
dev = pd.read_json("hf://datasets/G4KMU/t2-ragbench/" + splits["dev"], lines=True)
test = pd.read_json("hf://datasets/G4KMU/t2-ragbench/" + splits["test"], lines=True)

/home/felipe/financial-rag-benchmark/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train.head()

,id,context_id,split,question,program_answer,original_answer,context,file_name,company_name,report_year,company_sector
0,tatqa_train_0,d58da8b044c0221e4ad5fb3c60a50486,train,What was the amount of Income before income ta...,216.0,['$216.0'],The changes in AOCI associated with derivative...,raw/d58da8b044c0221e4ad5fb3c60a50486.pdf,carpenter-technology-corp,2019,Specialty Metals
1,tatqa_train_1,d58da8b044c0221e4ad5fb3c60a50486,train,What was the amount of Domestic Income before ...,140.3,['$140.3'],The changes in AOCI associated with derivative...,raw/d58da8b044c0221e4ad5fb3c60a50486.pdf,carpenter-technology-corp,2019,Specialty Metals
2,tatqa_train_2,d58da8b044c0221e4ad5fb3c60a50486,train,In the income before income taxes table for Ca...,2018.0,['2018'],The changes in AOCI associated with derivative...,raw/d58da8b044c0221e4ad5fb3c60a50486.pdf,carpenter-technology-corp,2019,Specialty Metals
3,tatqa_train_3,d58da8b044c0221e4ad5fb3c60a50486,train,What was the change in income before income ta...,-8.1,-8.1,The changes in AOCI associated with derivative...,raw/d58da8b044c0221e4ad5fb3c60a50486.pdf,carpenter-technology-corp,2019,Specialty Metals
4,tatqa_train_4,d58da8b044c0221e4ad5fb3c60a50486,train,What was the percentage change in Foreign inco...,-40.7,-40.7,The changes in AOCI associated with derivative...,raw/d58da8b044c0221e4ad5fb3c60a50486.pdf,carpenter-technology-corp,2019,Specialty Metals


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 9063 entries, 0 to 9062
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               9063 non-null   str  
 1   context_id       9063 non-null   str  
 2   split            9063 non-null   str  
 3   question         9063 non-null   str  
 4   program_answer   9063 non-null   str  
 5   original_answer  9063 non-null   str  
 6   context          9063 non-null   str  
 7   file_name        9063 non-null   str  
 8   company_name     9063 non-null   str  
 9   report_year      9063 non-null   int64
 10  company_sector   7649 non-null   str  
dtypes: int64(1), str(10)
memory usage: 39.3 MB


In [5]:
train["question"].loc[0]

'What was the amount of Income before income taxes for carpenter-technology-corp in the year ended June 30, 2019?'

In [6]:
train["program_answer"].loc[0]

'216.0'

In [7]:
train["original_answer"].loc[0]

"['$216.0']"

In [8]:
train["context"].loc[0]

'The changes in AOCI associated with derivative hedging activities during the years ended June 30, 2019, 2018 and 2017 were as follows:\n\n| ($ in millions) | 2019          | 2018       | 2017       |\n|-----------------|---------------|------------|------------|\n| Balance, beginning                       | $ 23.8       | $(2.3)     | $(21.8)    |\n| Cumulative adjustment upon adoption of ASU 2017-12 reclassified to reinvested earnings | (1.0)        | —          | —          |\n| Current period changes in fair value, net of tax | $(32.7)      | 26.9       | 5.8        |\n| Reclassification to earnings, net of tax | (4.9)        | (0.8)      | 13.7       |\n| Balance, ending                           | $(14.8)      | $ 23.8     | $(2.3)     |\n\n17. Income Taxes\n\nIncome before income taxes for the Company’s domestic and foreign operations was as follows:\n\n| ($ in millions) | Years Ended June 30, |\n|-----------------|----------------------|\n|                 | 2019  | 2018  | 201

In [9]:
train["file_name"].loc[0]

'raw/d58da8b044c0221e4ad5fb3c60a50486.pdf'

In [10]:
train["context_id"].describe()

count                                 9063
unique                                2172
top       9bd38f1be159f56aadd0153a76503bd5
freq                                     7
Name: context_id, dtype: object

# 02. Pipeline Setup

## Embeddings

In [11]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

In [27]:
embeddings = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project=PROJECT_ID,
    location=LOCATION
)

/tmp/ipykernel_2135727/2065440009.py:1: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(


In [23]:
train_contexts = train.drop_duplicates(subset=["context"], keep="first").reset_index(drop=True)

In [24]:
documents = []

for index, row in train_contexts.iterrows():
    doc = Document(
        page_content=row["context"],
        metadata={
            "file_name": row["file_name"],
            "context_id": row["split"]
        }
    )
    documents.append(doc)

In [36]:
vectorstore = None

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i+BATCH_SIZE]

    if vectorstore is None:
        vectorstore = FAISS.from_documents(
            batch,
            embeddings
        )
    else:
        vectorstore.add_documents(batch)

In [37]:
vectorstore.save_local(
    "vector_db"
)

In [ ]:
# vectorstore = FAISS.load_local(
#     "vector_db",
#     embeddings,
#     allow_dangerous_deserialization=True
# )

## RAG

In [38]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [43]:
def doc_helper(docs):
    return ". ".join(doc.page_content for doc in docs)

## Modelo

In [42]:
llm = ChatVertexAI(
    model="gemini-2.5-flash"
)

/tmp/ipykernel_2135727/366428109.py:1: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(
/tmp/ipykernel_2135727/366428109.py:1: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(


## Parser

In [57]:
class AnswerSchema(BaseModel):
    answer: str = Field(..., description="The answer to the question based on the provided context.")
    source: List[str] = Field(..., description="The list of retrieved contexts that were used to answer the question.")

In [58]:
parser = PydanticOutputParser(pydantic_object=AnswerSchema)

## Prompt

In [59]:
prompt = ChatPromptTemplate.from_template(
    """"
You are a helpful assistant.

Use ONLY the provided context.

Context:
{context}

Question: {question}

Format instructions:
{format_instructions}

Answer:
"""
)

## Chain

In [61]:
fillers = {
    "context": retriever | doc_helper,
    "question": RunnablePassthrough(),
    "format_instructions": lambda _ : parser.get_format_instructions()
}

chain = fillers | prompt | llm | parser

# Execution

In [62]:
entries = train["question"].tolist()[:10]

for entry in entries:
    response = chain.invoke(entry)
    print(response)

answer='The Income before income taxes for the year ended June 30, 2019, was $216.0 million.' source=['17. Income Taxes\nIncome before income taxes for the Company’s domestic and foreign operations was as follows:\n\n| ($ in millions) | Years Ended June 30, |\n|-----------------|----------------------|\n|                 | 2019  | 2018  | 2017  |\n| Domestic        |       |       |       |\n| $204.2          | $140.3| $56.0 |\n| Foreign         | 11.8  | 19.9  | 14.2  |\n| Income before income taxes | $216.0   | $160.2| $70.2 |']
answer='The Domestic Income before income taxes for Carpenter Technology Corp in Fiscal 2018 was $(245) million.' source=['|                | Fiscal 2019 (in millions) | Fiscal 2018 (in millions) | Fiscal 2017 (in millions) |\n|----------------|---------------------------|--------------------------|--------------------------|\n| U.S.           | $(216)                    | $(245)                   | $(273)                   |\n| Non-U.S.       | 2,147        

# Eval

In [63]:
answers = train["original_answer"].tolist()[:10]

In [64]:
entries

['What was the amount of Income before income taxes for carpenter-technology-corp in the year ended June 30, 2019?',
 'What was the amount of Domestic Income before income taxes for Carpenter Technology Corp in 2018?',
 "In the income before income taxes table for Carpenter Technology Corp's 2019 report, during which year did Foreign reach its highest value?",
 'What was the change in income before income taxes for Foreign operations from 2018 to 2019 for Carpenter Technology Corp?',
 'What was the percentage change in Foreign income before income taxes for carpenter-technology-corp from 2018 to 2019?',
 "In what month and year were the debt derivatives related to the repayment of the entire outstanding principal amount of Rogers Communications Inc.'s US$1.4 billion 6.8% senior notes settled, which were initially due?",
 'What was the fixed hedged rate for the April 30, 2019 issuance of senior notes?',
 'What was the decrease in the maturity coupon rate for the 2019 issuances from Apri

In [65]:
answers

["['$216.0']",
 "['$140.3']",
 "['2018']",
 '-8.1',
 '-40.7',
 "['August 2018']",
 "['4.173%']",
 '-0.18',
 '370',
 '1125']